In [1]:
from pyspark import SparkContext
from pyspark.streaming import StreamingContext

# Create a local StreamingContext with two working thread and batch interval of 5 seconds
# sparkContext를 호출하면서 local 서버에서 코어를 2개만 사용하겠다는 의미, 1은 안되는게 최소한 JVM이 2개가 돌아가야 하기 때문임 -> 소켓서버와 프로세스 서버
sc = SparkContext("local[2]", "NetworkWordCount")

# 10 sec interval
# 10초마다 한번씩 프로세스 진행, 10초마다 들어온 데이터를 자동으로 aggregate
ssc = StreamingContext(sc, 10)

# Create a DStream that will connect to hostname:port, like localhost:9999
# 연결한 netcat port 9999의 데이터를 읽어와서 각각의 라인으로 리스트를 형성
lines = ssc.socketTextStream("127.0.0.1", 9999)

# Split each line into words
# 읽어온 라인들을 flatMap을 통해서 split을 하고 하나하나를 튜플로 만듬
words = lines.flatMap(lambda line: line.split(" "))


# Count each word in each batch
# 각각의 word를 튜플로 만든 뒤 reduceByKey를 통해서 워드의 카운트를 계산
pairs = words.map(lambda word: (word, 1))
wordCounts = pairs.reduceByKey(lambda x, y: x + y)

# Print the first ten elements of each RDD generated in this DStream to the console
# pprint를 통해서 커맨드 라인에 결과를 보여줌
wordCounts.pprint()

ssc.start()             # Start the computation
# ssc.awaitTermination()  # Wait for the computation to terminate


/usr/local/spark/python/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


-------------------------------------------
Time: 2024-09-03 06:59:10
-------------------------------------------

-------------------------------------------
Time: 2024-09-03 06:59:20
-------------------------------------------

-------------------------------------------
Time: 2024-09-03 06:59:30
-------------------------------------------
('hello', 1)
('world', 1)

-------------------------------------------
Time: 2024-09-03 06:59:40
-------------------------------------------
('hello', 2)
('korea', 1)
('hi', 1)

-------------------------------------------
Time: 2024-09-03 06:59:50
-------------------------------------------
('hello', 1)
('bye', 1)

-------------------------------------------
Time: 2024-09-03 07:00:00
-------------------------------------------

-------------------------------------------
Time: 2024-09-03 07:00:10
-------------------------------------------

-------------------------------------------
Time: 2024-09-03 07:00:20
---------------------------------------